# Session 11: 位置制御 — 座標変換と外乱抑制
# Session 11: Position Control — Coordinate Transform and Disturbance Rejection

## 目標 / Objective

NED/Body 座標変換と 4 段カスケードの全体像を理解する。

Understand NED/Body coordinate transformation and the full 4-stage cascade.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| 4段カスケード | 位置 → 速度 → 角度 → 角速度 |
| NED/Body 変換 | 位置誤差を機体座標に変換 |
| 外乱抑制 | 風外乱下での定位精度 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Ready! / 準備完了！")

## 1. 4段カスケード全体像 / 4-Stage Cascade Overview

```
位置目標 → [位置PID] → 速度目標 → [速度PID] → 角度目標 → [角度PID] → 角速度目標 → [レートPID] → モーター
   ↑                                                                                              │
   └────────────────────────── センサフュージョン (ESKF) ←──────────────────────────────────────────┘
```

### 帯域設計 / Bandwidth Design

| ループ | 帯域 (rad/s) | 制御周期 |
|--------|-------------|---------|  
| 位置 | ~1 | 50 Hz |
| 速度 | ~5 | 50 Hz |
| 角度 | ~15 | 400 Hz |
| 角速度 | ~50 | 400 Hz |

各段で帯域が 3〜5 倍ずつ速くなる。

## 2. NED/Body 座標変換 / NED/Body Coordinate Transform

位置誤差は NED 座標系で計算されますが、推力方向は機体座標系です。

$$\begin{bmatrix} e_x^{body} \\ e_y^{body} \end{bmatrix} = \begin{bmatrix} \cos\psi & \sin\psi \\ -\sin\psi & \cos\psi \end{bmatrix} \begin{bmatrix} e_N \\ e_E \end{bmatrix}$$

In [ ]:
# Visualize NED to Body transformation
# NED → Body 座標変換の可視化

def ned_to_body(error_ned, yaw_deg):
    """Transform NED error to body frame."""
    psi = np.radians(yaw_deg)
    R = np.array([
        [np.cos(psi), np.sin(psi)],
        [-np.sin(psi), np.cos(psi)],
    ])
    return R @ error_ned

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

error_ned = np.array([1.0, 0.5])  # 1m north, 0.5m east

for ax, yaw in zip(axes, [0, 45, 90]):
    error_body = ned_to_body(error_ned, yaw)
    
    # Plot NED error
    ax.arrow(0, 0, error_ned[0], error_ned[1], head_width=0.05,
             color='blue', alpha=0.5, label='NED error')
    
    # Plot body axes
    psi = np.radians(yaw)
    ax.arrow(0, 0, np.cos(psi)*0.5, np.sin(psi)*0.5, head_width=0.03,
             color='red', label='Body X (forward)')
    ax.arrow(0, 0, -np.sin(psi)*0.5, np.cos(psi)*0.5, head_width=0.03,
             color='green', label='Body Y (right)')
    
    # Plot body error
    ax.arrow(0, 0, error_body[0]*np.cos(psi) - error_body[1]*np.sin(psi),
             error_body[0]*np.sin(psi) + error_body[1]*np.cos(psi),
             head_width=0.05, color='purple', alpha=0.5, linestyle='--',
             label=f'Body error: ({error_body[0]:.2f}, {error_body[1]:.2f})')
    
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(f'Yaw = {yaw}°')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('North / 北')
    ax.set_ylabel('East / 東')

fig.suptitle('NED to Body Transform / NED → Body 座標変換', fontsize=14)
fig.tight_layout()
plt.show()

## 3. 2D 位置制御シミュレーション / 2D Position Control Simulation

In [ ]:
# Simple 2D position control simulation
# 簡易 2D 位置制御シミュレーション

import sys
from pathlib import Path
_vpython_dir = Path().absolute().parent.parent.parent / 'simulator' / 'vpython' / 'control'
if str(_vpython_dir) not in sys.path:
    sys.path.insert(0, str(_vpython_dir))

from pid import PID

dt = 0.02  # 50Hz position loop
duration = 20.0
n = int(duration / dt)

# Waypoints (m)
waypoints = [(1, 0), (1, 1), (0, 1), (0, 0)]
wp_idx = 0
wp_threshold = 0.1  # m

# State
x, y = 0.0, 0.0
vx, vy = 0.0, 0.0

# Controllers
pos_pid_x = PID(Kp=2.0, Ti=2.0, output_min=-0.5, output_max=0.5)
pos_pid_y = PID(Kp=2.0, Ti=2.0, output_min=-0.5, output_max=0.5)
vel_pid_x = PID(Kp=5.0, Ti=0.5, output_min=-2.0, output_max=2.0)
vel_pid_y = PID(Kp=5.0, Ti=0.5, output_min=-2.0, output_max=2.0)

# Logging
t_log = np.zeros(n)
x_log = np.zeros(n)
y_log = np.zeros(n)

rng = np.random.default_rng(42)

for i in range(n):
    t = i * dt
    t_log[i] = t
    x_log[i] = x
    y_log[i] = y
    
    # Target
    if wp_idx < len(waypoints):
        tx, ty = waypoints[wp_idx]
        dist = np.sqrt((tx-x)**2 + (ty-y)**2)
        if dist < wp_threshold:
            wp_idx += 1
            if wp_idx < len(waypoints):
                tx, ty = waypoints[wp_idx]
    else:
        tx, ty = 0.0, 0.0
    
    # Position loop -> velocity setpoint
    vx_sp = pos_pid_x.update(tx, x, dt)
    vy_sp = pos_pid_y.update(ty, y, dt)
    
    # Velocity loop -> acceleration command
    ax_cmd = vel_pid_x.update(vx_sp, vx, dt)
    ay_cmd = vel_pid_y.update(vy_sp, vy, dt)
    
    # Wind disturbance (random, small)
    wind_x = rng.normal(0, 0.1)
    wind_y = rng.normal(0, 0.1)
    
    # Dynamics
    vx += (ax_cmd + wind_x) * dt
    vy += (ay_cmd + wind_y) * dt
    x += vx * dt
    y += vy * dt

# Plot trajectory
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# XY trajectory
ax1.plot(x_log, y_log, 'b-', linewidth=1.0, label='Actual / 実際')
wp_x = [p[0] for p in waypoints] + [waypoints[0][0]]
wp_y = [p[1] for p in waypoints] + [waypoints[0][1]]
ax1.plot(wp_x, wp_y, 'r--', linewidth=1.5, label='Waypoints / 目標')
ax1.plot(0, 0, 'go', markersize=10, label='Start')
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_title('Position Control with Wind / 風外乱下の位置制御')
ax1.set_aspect('equal')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Time series
ax2.plot(t_log, x_log, 'b-', label='X')
ax2.plot(t_log, y_log, 'r-', label='Y')
ax2.set_xlabel('Time (s) / 時間')
ax2.set_ylabel('Position (m) / 位置')
ax2.set_title('Position vs Time / 位置 vs 時間')
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 4. 考察課題 / Discussion Questions

1. **座標変換の重要性**: ヨー角が 90° のとき、「北に進め」を機体にどう伝えるか？
2. **風外乱**: 一定方向の風と突風（ランダム）、それぞれに対するフィードバックの効果は？
3. **4段カスケードの限界**: 各ループに遅延があるとき、全体の帯域はどう制限されるか？

---

1. **Coordinate Transform**: When yaw is 90°, how do you command "go north" in body frame?
2. **Wind Disturbance**: How does feedback handle constant wind vs gusts (random)?
3. **4-Stage Cascade Limit**: With delay in each loop, how is overall bandwidth limited?

In [ ]:
# Your experiments here / ここで実験してみてください
